In [ ]:
%load_ext autoreload
%autoreload 2

# WAM dreamzero data viz

Loads one random human and one random robot episode from a locally-staged Zarr
folder using the exact keymap + transform pipeline used by our WAM dreamzero
training runs (`Mecka.get_wam_*` for human_bimanual, `Eva.get_wam_*` for
eva_bimanual, both wired via `LocalWamEpisodeResolver` and wrapped in
`WamMultiDataset` which skips quantile bounds calibration). This mirrors
`hydra_configs/data/human_wam_v2.yaml` (human) and `evaluator/viz/wam_cartesian_eva.yaml` (robot).

The batch has a video *clip* under `observations.images.front_img_1`
(shape `(B, T, C, H, W)`, `T = cam_horizon = 17`) plus the 16-step
`actions_cartesian` chunk. Both embodiments emit the same 12D action layout
(`[L xyz ypr, R xyz ypr]`), so the same viz code renders both.

In [ ]:
import pathlib
import random

import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch
import zarr

from egomimic.rldb.embodiment.eva import Eva
from egomimic.rldb.embodiment.human import Mecka
from egomimic.rldb.filters import DatasetFilter
from egomimic.rldb.zarr.zarr_dataset_wam import (
    LocalWamEpisodeResolver,
    WamMultiDataset,
)

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
TEMP_DIR = "/storage/project/r-dxu345-0/shared/arc_test"

In [ ]:
# WAM dreamzero pipeline — one loader per embodiment, matching the
# ``embodiment == '<name>'`` filter used by our production configs
# (human_wam_v2.yaml, mecka_wam.yaml, aria_wam.yaml). We enumerate matching
# episodes on disk and randomly pick one so re-running the notebook picks a
# different clip; the WAM keymap alignment (cam_horizon=17, action_horizon=16,
# state_horizon=4) is the exact one used at train + eval time.
EMBODIMENT_SPECS = {
    "human_bimanual": {
        "keymap_fn": Mecka.get_wam_keymap,
        "transform_fn": Mecka.get_wam_transform_list,
        "viz_cls": Mecka,
    },
    "eva_bimanual": {
        "keymap_fn": Eva.get_wam_keymap,
        "transform_fn": Eva.get_wam_transform_list,
        "viz_cls": Eva,
    },
}

# Val-video container config — copied from WAMEvalVideo (eval_wam.py):
# 30 fps container, each downsampled pixel frame held FRAME_REPEAT output
# frames so content updates at VAL_FPS/FRAME_REPEAT = 5 fps.
VAL_FPS = 30
FRAME_REPEAT = 6


def _list_episode_hashes(folder_path, embodiment):
    """Enumerate on-disk episode hashes matching ``embodiment`` — same metadata
    read that ``LocalEpisodeResolver`` does, without needing to build a dataset."""
    hashes = []
    for p in sorted(pathlib.Path(folder_path).iterdir()):
        if not p.is_dir():
            continue
        try:
            attrs = dict(zarr.open_group(str(p), mode="r").attrs)
        except Exception:
            continue
        if attrs.get("embodiment") == embodiment:
            hashes.append(p.name)
    return hashes


def build_random_wam_loader(folder_path, embodiment, seed=None):
    spec = EMBODIMENT_SPECS[embodiment]
    hashes = _list_episode_hashes(folder_path, embodiment)
    if not hashes:
        raise RuntimeError(f"No {embodiment} episodes found in {folder_path}")
    rng = random.Random(seed)
    chosen = rng.choice(hashes)
    print(f"[{embodiment}] {len(hashes)} episode(s) available; picked {chosen!r}")

    resolver = LocalWamEpisodeResolver(
        folder_path=folder_path,
        key_map=spec["keymap_fn"](
            cam_horizon=17, action_horizon=16, state_horizon=4
        ),
        transform_list=spec["transform_fn"](),
    )
    filters = DatasetFilter(
        filter_lambdas=[
            f"lambda row: row['embodiment'] == '{embodiment}' "
            f"and row['episode_hash'] == '{chosen}'"
        ]
    )
    ds = WamMultiDataset._from_resolver(resolver, filters=filters, mode="total")
    return torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False), spec["viz_cls"]


def render_wam_val_video(loader, viz_cls, max_clips=10):
    """Reproduces the WAM validation-video animation exactly (matches
    ``WAMEvalVideo.compute_metrics_and_viz`` inner loop):

      - ``VAL_FPS`` = 30 (container), each pixel frame held ``FRAME_REPEAT``
        output frames -> content updates at 5 fps.
      - Anchor frame (index 0) skipped; ``T-1`` pixel transitions per clip.
      - Action-arrow trail origin snaps to ``pixel_idx * FRAME_REPEAT`` so the
        arrow starts at the hand for that pixel, and shrinks by one arrow per
        intra-pixel tick (j) as the countdown until the next pixel update.
      - No predictions available in the notebook -> we only draw GT (the green
        half of ``viz_gt_preds``).
    """
    all_frames = []
    for clip_i, batch in enumerate(loader):
        clip = batch["observations.images.front_img_1"][0]       # (T, C, H, W)
        actions = batch["actions_cartesian"][0].cpu().numpy()    # (n_actions, 12)
        K = batch["intrinsics"][0].cpu().numpy()                 # (3, 4)
        T_pix = clip.shape[0]
        n_actions = actions.shape[0]
        total_out_frames = (T_pix - 1) * FRAME_REPEAT
        for f in range(total_out_frames):
            pixel_idx = 1 + f // FRAME_REPEAT
            pixel_action_start = pixel_idx * FRAME_REPEAT
            j = f % FRAME_REPEAT
            start = min(pixel_action_start + j, n_actions - 1)
            end = min((pixel_idx + 1) * FRAME_REPEAT, n_actions)
            if end <= start:
                end = min(start + 1, n_actions)
            all_frames.append(
                viz_cls.viz(
                    image=clip[pixel_idx].cpu().numpy(),
                    viz_data=actions[start:end],
                    mode="traj",
                    intrinsics=K,
                    color="Greens",
                )
            )
        if clip_i + 1 >= max_clips:
            break
    return all_frames

## Random `human_bimanual` episode
Universal WAM embodiment (aria/mecka data). Actions/state transformed into
head frame via `obs_head_pose`.

In [ ]:
human_loader, human_viz = build_random_wam_loader(TEMP_DIR, "human_bimanual")

# Inspect one batch.
human_batch = next(iter(human_loader))
for k, v in human_batch.items():
    print(f"  {k:45s} {tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__}")

# Val-video: 30 fps container, content updates at 5 fps (FRAME_REPEAT=6 hold).
# max_clips=1 keeps this light — one clip is 17 pixel frames * 6 = 96 rendered
# output frames. Bump it if you want a longer sample.
mpy.show_video(render_wam_val_video(human_loader, human_viz, max_clips=1), fps=VAL_FPS)

## Random `eva_bimanual` episode
Robot data — actions/state projected to camera frame via `Eva.EXTRINSICS`
(no head pose needed). Grippers are dropped so the 12D `[L xyz ypr, R xyz ypr]`
layout matches human_bimanual and the same renderer works.

In [ ]:
robot_loader, robot_viz = build_random_wam_loader(TEMP_DIR, "eva_bimanual")

# Inspect one batch.
robot_batch = next(iter(robot_loader))
for k, v in robot_batch.items():
    print(f"  {k:45s} {tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__}")

mpy.show_video(render_wam_val_video(robot_loader, robot_viz, max_clips=1), fps=VAL_FPS)